# Lodestar Demo Notebook

This notebook demonstrates the full Lodestar pipeline: from policy ingestion to cultural fit scoring to implementation blueprint generation.

We'll run a complete analysis of **Detroit's urban mobility challenge**, matching it against the global policy knowledge base and generating a culturally-adapted implementation blueprint.

**Prerequisites:**
```bash
pip install -e ".[dev]"   # from the lodestar project root
cp .env.example .env      # add your ANTHROPIC_API_KEY
```

## Step 1: Setup and Imports

Import the key components from each layer of the Lodestar pipeline.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Engine layer
from lodestar.culture.adaptor import CulturalAdaptor

# Culture layer
from lodestar.culture.profiles import CULTURAL_PROFILES, get_profile
from lodestar.culture.scorer import CulturalScorer
from lodestar.engine.embeddings import EmbeddingsDB
from lodestar.engine.matcher import PolicyMatcher
from lodestar.engine.risk import RiskAnalyzer
from lodestar.engine.simulator import ImpactSimulator
from lodestar.output.brief import generate_markdown_brief

# Output layer

print(f"Lodestar ready. ANTHROPIC_API_KEY set: {bool(os.getenv('ANTHROPIC_API_KEY'))}")
print(f"Countries in cultural database: {len(CULTURAL_PROFILES)}")

## Step 2: Initialize the Vector Database

Lodestar uses ChromaDB to store semantic embeddings of the policy knowledge base. On first run, we need to ingest the seed policies from `seed_policies.json`.

The `EmbeddingsDB` class handles persistence — subsequent runs load from the existing `.chroma/` directory without re-embedding.

In [ ]:
import json

# Initialize the ChromaDB vector database
db = EmbeddingsDB(persist_dir="../.chroma")
db.initialize_db()

# Load seed policies from JSON
seed_path = Path("../lodestar/data/seed_policies.json")
with open(seed_path) as f:
    policies = json.load(f)

print(f"Loaded {len(policies)} policies from seed_policies.json")

# Ingest all policies into ChromaDB
for policy in policies:
    db.add_policy(policy)
    print(f"  ✓ {policy['name']} ({policy['country']}, {policy['domain']})")

print(f"\nVector database ready. Total policies indexed: {len(db.get_all_policies())}")

## Step 3: Inspect the Cultural Profile Database

Before running the analysis, let's look at the cultural profiles for the countries involved in our query: the **United States** (target) and **Colombia** (source country for Medellín).

This is the data that drives the Cultural Fit Score.

In [ ]:
import pandas as pd

# Fetch profiles for comparison
us_profile = get_profile("US")
co_profile = get_profile("CO")  # Colombia
sg_profile = get_profile("SG")  # Singapore (for reference)

# Build comparison DataFrame
dimensions = [
    "power_distance",
    "individualism",
    "uncertainty_avoidance",
    "long_term_orientation",
    "institutional_trust",
    "tech_adoption_index",
]

comparison = pd.DataFrame(
    {
        "Colombia (Medellín)": [co_profile[d] for d in dimensions],
        "United States (Detroit)": [us_profile[d] for d in dimensions],
        "Singapore (reference)": [sg_profile[d] for d in dimensions],
    },
    index=dimensions,
)

print("Cultural Dimension Comparison")
print("=" * 55)
print(comparison.to_string())
print("\nNote: Higher institutional_trust and lower power_distance")
print("generally correlate with easier policy transfer.")

## Step 4: Semantic Policy Matching

The `PolicyMatcher` constructs a rich query embedding and searches the ChromaDB vector store for the most similar policies.

We're asking: **"What global urban mobility programs are most analogous to Detroit's challenge?"**

In [ ]:
# Initialize the policy matcher
matcher = PolicyMatcher(db)

# Run semantic matching
city = "Detroit"
domain = "urban_mobility"
country = "US"
context = "Post-industrial city with 25% poverty rate, high transit dependency, seeking to connect peripheral neighborhoods to downtown employment core"

matches = matcher.match(
    city=city, domain=domain, country=country, context=context, n_results=3
)

print(f"Top {len(matches)} matches for '{city}' + '{domain}':\n")
for i, match in enumerate(matches, 1):
    print(f"{i}. {match['name']}")
    print(f"   Country: {match['metadata'].get('country', 'Unknown')}")
    print(f"   Similarity: {match['similarity_score']:.3f}")
    print(f"   Domain: {match['metadata'].get('domain', 'Unknown')}")
    print()

## Step 5: Cultural Fit Scoring

For each matched policy, compute the **Cultural Fit Score** between the source country and the target (US/Detroit).

The score is a weighted composite across six Hofstede dimensions plus institutional metrics. Higher = better fit, less adaptation required.

In [ ]:
scorer = CulturalScorer()

print("Cultural Fit Scores — Source Country → United States\n")
print("-" * 60)

# Score the top match (Medellín / Colombia → US)
top_match_country = matches[0]["metadata"].get("country_code", "CO")

# Score all unique source countries
seen_countries = set()
for match in matches:
    source_code = match["metadata"].get("country_code", "CO")
    if source_code in seen_countries:
        continue
    seen_countries.add(source_code)

    try:
        result = scorer.score(source_code, "US")
        print(f"\n{result['source_country']} → {result['target_country']}")
        print(f"  Score: {result['score']}/100  Grade: {result['grade']}")
        print(f"  Summary: {result['summary'].split('—')[1].strip()}")

        print("  Dimension scores:")
        for dim, score in result["dimension_scores"].items():
            bar = "█" * int(score / 10) + "░" * (10 - int(score / 10))
            print(f"    {dim:<28} {bar}  {score:.0f}/100")

        if result["adaptation_notes"]:
            print("  Adaptation notes:")
            for note in result["adaptation_notes"]:
                print(f"    • {note[:90]}..." if len(note) > 90 else f"    • {note}")
    except ValueError as e:
        print(f"  {source_code}: {e}")

## Step 6: Risk Analysis

The `RiskAnalyzer` computes a composite risk score using cultural gap, governance capacity, and domain-specific risk factors.

Risk is not a reason to avoid a program — it's information about what design choices will determine success or failure.

In [ ]:
analyzer = RiskAnalyzer()

# Compute Cultural Fit Score for the primary match
fit_result = scorer.score("CO", "US")  # Colombia → US

# Get governance score (World Bank WGI percentile rank, 0-100)
# Using approximate value; in production this comes from WorldBankClient
governance_score = us_profile.get("regulatory_quality", 75)

risk_result = analyzer.analyze(
    matches=matches,
    fit_score=fit_result["score"],
    governance_score=governance_score,
    domain=domain,
    target_country="US",
)

print("Risk Assessment — Detroit Urban Mobility Program")
print("=" * 50)
print(f"Overall Risk Score: {risk_result['overall_score']}/100")
print(f"Risk Level: {risk_result['risk_level']}")
print()
print("Risk Factors:")
for factor in risk_result.get("risk_factors", []):
    print(f"  • {factor}")
print()
print("Mitigation Strategies:")
for strategy in risk_result.get("mitigation_strategies", []):
    print(f"  ✓ {strategy}")

## Step 7: Impact Simulation (Claude)

The `ImpactSimulator` sends matched policies, target context, and Cultural Fit Score to **Claude Sonnet 4.6** to generate predicted outcomes with confidence ranges.

This step requires a valid `ANTHROPIC_API_KEY` in your `.env` file.

In [ ]:
if not os.getenv("ANTHROPIC_API_KEY"):
    print("⚠️  ANTHROPIC_API_KEY not set. Skipping live simulation.")
    print("   Set ANTHROPIC_API_KEY in .env to enable impact simulation.")
    simulation_result = None
else:
    simulator = ImpactSimulator()

    target_context = f"""
    City: {city}, {country}
    Domain: {domain}
    Context: {context}
    Cultural Fit Score: {fit_result["score"]}/100 ({fit_result["grade"]})
    """

    print("Running impact simulation via Claude Sonnet 4.6...")
    simulation_result = simulator.simulate(
        matches=matches,
        target_context=target_context,
        cultural_fit_score=fit_result["score"],
    )

    print(
        f"\nSimulation complete. Confidence: {simulation_result.get('confidence_score', 0):.0%}"
    )
    print("\nPredicted Outcomes:")
    for outcome in simulation_result.get("predicted_outcomes", []):
        print(
            f"  • {outcome['outcome']}: {outcome['value_range']} ({outcome['timeline']})"
        )

    print(f"\nBull case: {simulation_result.get('bull_case', 'N/A')[:120]}...")
    print(f"\nBear case: {simulation_result.get('bear_case', 'N/A')[:120]}...")
    print(f"\nRecommended pilot: {simulation_result.get('recommended_pilot', 'N/A')}")

## Step 8: Cultural Adaptation Recommendations

The `CulturalAdaptor` translates the Cultural Fit Score into concrete, actionable recommendations: what to keep from the source model, what to redesign, what to avoid, and what quick wins to pursue in the first 6 months.

In [ ]:
adaptor = CulturalAdaptor()

# Use the top matched policy for adaptation recommendations
top_policy = matches[0]

adaptation = adaptor.adapt(
    policy=top_policy, cultural_fit=fit_result, target_country_code="US"
)

print(f"Cultural Adaptation — {top_policy['name']} → {city}")
print("=" * 60)
print(f"Adaptation Level: {adaptation['adaptation_level']}")
print(f"Implementation Approach: {adaptation['implementation_approach']}")
print(f"Communication Strategy: {adaptation['communication_strategy']}")
print()

print("Adapted Elements:")
for element in adaptation.get("adapted_elements", []):
    print(f"  ↻ {element}")

print()
print("Quick Wins (0–6 months):")
for win in adaptation.get("quick_wins", []):
    print(f"  ✓ {win}")

print()
print("Avoid List:")
for avoid in adaptation.get("avoid_list", []):
    print(f"  ✗ {avoid}")

## Step 9: Generate the Full Implementation Brief

Assemble all layer outputs into a `LodestarBrief` and render it as a formatted Markdown document.

This is the output that would be saved to a `.md` or `.pdf` file via the CLI (`lodestar analyze --output brief.md`).

In [ ]:
from IPython.display import Markdown, display

# Assemble the brief data dict
brief_data = {
    "city": city,
    "country": country,
    "domain": domain,
    "matches": matches,
    "cultural_fit": fit_result,
    "risk_assessment": risk_result,
    "simulation": simulation_result,
    "adaptation": adaptation,
}

# Generate the Markdown brief
markdown_output = generate_markdown_brief(brief_data)

# Display a preview (first 3000 characters)
print(f"Brief generated: {len(markdown_output):,} characters")
print("\n" + "=" * 60)
print("PREVIEW (first 3000 chars):")
print("=" * 60 + "\n")
display(Markdown(markdown_output[:3000] + "\n\n*[... truncated for preview ...]*"))

## Step 10: Save the Brief

Save the complete brief as a Markdown file. You can also export to PDF using `lodestar/output/pdf.py`.

In [ ]:
output_path = Path(f"../examples/demo_output_{city.lower()}_{domain}.md")

with open(output_path, "w") as f:
    f.write(markdown_output)

print(f"Brief saved to: {output_path}")
print(f"File size: {output_path.stat().st_size:,} bytes")
print()
print("To export as PDF:")
print("  from lodestar.output.pdf import markdown_to_pdf")
print(
    f"  markdown_to_pdf(markdown_output, 'brief_{city.lower()}.pdf', title='Lodestar: {city} {domain}')"
)

## Appendix: Batch Analysis

Run multiple analyses in sequence to compare Cultural Fit Scores across different city-domain pairs.

In [ ]:
# Batch comparison: Cultural Fit Scores for a selection of policy transfers
transfers = [
    ("SG", "NG", "Singapore → Nigeria"),
    ("EE", "BR", "Estonia → Brazil"),
    ("CO", "US", "Colombia → United States"),
    ("DK", "SE", "Denmark → Sweden"),  # high-fit baseline
    ("SG", "US", "Singapore → United States"),
    ("RW", "NG", "Rwanda → Nigeria"),
]

results = []
for source, target, label in transfers:
    try:
        r = scorer.score(source, target)
        results.append(
            {
                "Transfer": label,
                "Score": r["score"],
                "Grade": r["grade"],
                "Notes": len(r["adaptation_notes"]),
            }
        )
    except ValueError:
        results.append({"Transfer": label, "Score": "N/A", "Grade": "N/A", "Notes": 0})

df = pd.DataFrame(results).sort_values("Score", ascending=False)
print("Cultural Fit Score Comparison")
print("=" * 55)
print(df.to_string(index=False))
print()
print("Grade scale: A (80+) | B (65-79) | C (50-64) | D (35-49) | F (<35)")
print("Notes = number of specific adaptation recommendations generated")

## Next Steps

- **Add policies:** See [docs/adding-policies.md](../docs/adding-policies.md) for the schema and submission process
- **Extend country profiles:** Edit `lodestar/culture/profiles.py` to add new countries
- **Use the REST API:** `make api` → `POST /analyze` with the same parameters
- **Export PDFs:** `from lodestar.output.pdf import markdown_to_pdf`
- **Read the architecture:** [docs/architecture.md](../docs/architecture.md)

---

*Lodestar v0.1.0 · Built by Jean-Paul Daum · [jeanpauldaum.com](https://jeanpauldaum.com)*